<a href="https://colab.research.google.com/github/ravindyaparami/Statistical-Learning-e20056/blob/main/E20056_Bayesian_Inference_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

An online learning platform presents a user with a sequence of $n$ multiple-choice questions **one at a time**. Each question is either answered correctly or incorrectly, allowing the platform to update its estimate of the user's ability dynamically after every response.

Let $Y_i$ denote the user's response to the $i$-th item encountered:

$$Y_i=
\begin{cases}
1, & \text{if the user answers item } i \text{ correctly},\\
0, & \text{if the user answers item } i \text{ incorrectly}.
\end{cases}$$

The platform assumes that the probability of a correct response is governed by a two-parameter logistic (2PL) item response model. Specifically, conditional on the user's latent ability parameter $\Theta=\theta$, the response probability for item $i$ is:

$$P(Y_i=1\mid \Theta=\theta)=p_i(\theta)=\frac{1}{1+e^{-a_i(\theta-b_i)}},$$

where $a_i>0$ is the known discrimination parameter, and $b_i$ is the known difficulty parameter of item $i$.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed responses** up to the current step $k$ (where $1 \le k \le n$).

Before observing any responses, the platform initializes the user's latent ability estimate with a standard normal prior distribution:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta^2}{2}\right) \quad \text{implying} \quad \Theta \sim \mathscr{N}(0,1).$$

As the user progresses, the posterior distribution at step $k-1$ serves as the prior distribution for step $k$.

---

### Tasks

1. **Visualizing the Mechanics:** Plot $P(Y_i=1\mid \Theta=\theta)$ vs $\theta$ using Plotly for two distinct values of $a_i$, where one of those $a_i$ values is paired with three different difficulty values of $b_i$. Interpret how moving $b_i$ shifts the curve horizontally.
2. **Sequential Likelihood Contribution:** Write down the likelihood contribution $L(y_k \mid \theta)$ of a *single* new response $y_k$ at step $k$, given the latent ability $\theta$. Then, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.
3. **Mathematical Formulation of the Running Update:** Write down the recursive relationship for the posterior density at step $k$, denoted $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, up to a proportionality constant, using the prior state $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ and the new observation $y_k$.
4. **Dynamic Shifting:** Explain how a correct answer ($y_k = 1$) to a highly difficult item (large $b_k$) mathematically shifts the peak of the running posterior density distribution relative to the previous step.
5. **Tracking Certainty and Sharpness:** Explain how the discrimination parameter $a_k$ of the current item alters the variance (or "sharpness") of the distribution during a running update. What happens when $a_k$ is very large versus very small?
6. **Numerical Implementation of a Running Grid:** Describe a algorithmic approach to numerically approximate and maintain this running posterior density function on a fixed grid of $\theta$-values. Explicitly state how you would perform the sequential normalization step computationally after an item is answered.


7. **Evaluating Convergence over the Timeline:** Suppose the user's true, hidden latent ability is $\theta_{\text{true}} = 0.75$. Write a Python script that extends your previous grid simulation to track the performance of the running estimators over a sequence of $n = 20$ items.
* **Simulate Responses:** Dynamically generate the user's responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against the true response probability $p_k(\theta_{\text{true}})$. Give each item a random difficulty $b_k \sim \mathscr{N}(0, 1)$ and a random discrimination $a_k \sim \text{Uniform}(0.5, 2.0)$.
* **Track Estimators:** At each step $k$, calculate and store the running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$) and the running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$) estimate.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $20$. Add a static horizontal reference line at $y = 0.75$ representing $\theta_{\text{true}}$.
* **Analysis:** Briefly explain how the distance between your estimators and $\theta_{\text{true}}$ changes as $k$ increases, and interpret what this implies about the platform's confidence in its measurement.


#Answers

In [1]:
#1
import numpy as np
import plotly.graph_objects as go
# Ability range
theta = np.linspace(-4, 4, 500)

def probability_correct(theta, a, b):
  return 1 / (1 + np.exp(-a * (theta - b)))

fig = go.Figure()

# Same discrimination, different difficulties
a1 = 1.0

for b in [-1.0, 0.0, 1.0]:
  fig.add_trace(
      go.Scatter(
          x=theta,
          y=probability_correct(theta, a1, b),
          mode='lines',
          name=f'a={a1}, b={b}'
      )
  )

# A different discrimination value
a2 = 2.0
b2 = 0.0

fig.add_trace(
    go.Scatter(
        x=theta,
        y=probability_correct(theta, a2, b2),
        mode='lines',
        name=f'a={a2}, b={b2}' ) )

fig.update_layout(
    title='2PL Item Response Curves',
    xaxis_title='Ability θ',
    yaxis_title='P(Yᵢ = 1 | Θ = θ)',
    template='plotly_white' )

fig.show()

Interpretation

The difficulty parameter $b_i$ controls the horizontal position of the item response curve.

At

$$
\theta=b_i,
$$

we have

$$
p_i(\theta)=0.5.
$$

Therefore:

Increasing $b_i$ shifts the curve to the right, meaning that a higher ability is required to obtain the same probability of answering correctly.
Decreasing $b_i$ shifts the curve to the left, meaning that the item is easier.
The discrimination parameter $a_i$ controls the steepness of the curve. A larger $a_i$ produces a steeper transition around $\theta=b_i$.
2. Sequential Likelihood Contribution

For a new binary response

$$
y_k\in{0,1},
$$

the likelihood contribution of the $k$-th response is the Bernoulli likelihood
$$
[p_k(\theta)]^{y_k}
[1-p_k(\theta)]^{1-y_k},
$$

where
$$
\frac{1}{1+\exp[-a_k(\theta-b_k)]}.
$$

Therefore:

If $y_k=1$,

$$
L(y_k\mid\theta)=p_k(\theta).
$$

If $y_k=0$,

$$
L(y_k\mid\theta)=1-p_k(\theta).
$$

Assuming the responses are conditionally independent given $\Theta=\theta$, the joint likelihood for the running response history

$$
y^{(k)}=(y_1,\ldots,y_k)
$$

is
$$
\prod_{i=1}^{k}
[p_i(\theta)]^{y_i}
[1-p_i(\theta)]^{1-y_i}.
$$

3. Mathematical Formulation of the Running Bayesian Update

At step $k-1$, suppose the current posterior density is

$$
f_{\Theta\mid Y^{(k-1)}}
\left(
\theta\mid y^{(k-1)}
\right).
$$

This posterior becomes the prior for the next update.

After observing the new response $y_k$, Bayes' theorem gives

$$
f_{\Theta\mid Y^{(k)}}
\left(
\theta\mid y^{(k)}
\right)
\propto
L(y_k\mid\theta)
f_{\Theta\mid Y^{(k-1)}}
\left(
\theta\mid y^{(k-1)}
\right).
$$

Substituting the Bernoulli likelihood,

$$
f_{\Theta\mid Y^{(k)}}
\left(
\theta\mid y^{(k)}
\right)
\propto
[p_k(\theta)]^{y_k}
[1-p_k(\theta)]^{1-y_k}
f_{\Theta\mid Y^{(k-1)}}
\left(
\theta\mid y^{(k-1)}
\right).
$$

The fully normalized update is
$$
\frac{
[p_k(\theta)]^{y_k}
[1-p_k(\theta)]^{1-y_k}
f_{\Theta\mid Y^{(k-1)}}
(\theta\mid y^{(k-1)})
}{
\int
[p_k(u)]^{y_k}
[1-p_k(u)]^{1-y_k}
f_{\Theta\mid Y^{(k-1)}}
(u\mid y^{(k-1)})
,du
}.
$$

Thus, every response modifies the current posterior, and this updated posterior is carried forward to the next question.

4. Dynamic Shifting After a Correct Answer to a Difficult Item

Suppose the user correctly answers a highly difficult item, so

$$
y_k=1
$$

and $b_k$ is large.

For a correct response, the update becomes

$$
f_{\Theta\mid Y^{(k)}}(\theta\mid y^{(k)})
\propto
p_k(\theta)
f_{\Theta\mid Y^{(k-1)}}(\theta\mid y^{(k-1)}).
$$

For a difficult item,
$$
\frac{1}{1+\exp[-a_k(\theta-b_k)]}
$$

is small for low values of $\theta$ and becomes large only for sufficiently high values of $\theta$.

Therefore, multiplying the previous posterior by $p_k(\theta)$:

strongly reduces posterior density at low ability values,
gives relatively more weight to high ability values.

Consequently, the posterior peak generally shifts toward larger values of $\theta$.

The effect can also be seen from the log-likelihood contribution for a correct response:

$$
\log L(y_k=1\mid\theta)=\log p_k(\theta).
$$

Its derivative is
$$
a_k[1-p_k(\theta)]>0.
$$

Thus, a correct answer contributes a positive upward push to the estimated ability.

A correct response to a very difficult item is particularly strong evidence of high ability because such a response would have been unlikely under low values of $\theta$.

5. Effect of the Discrimination Parameter on Posterior Sharpness

The discrimination parameter $a_k$ determines how strongly the probability of a correct response changes with ability.

The slope of the 2PL response function is
$$
a_kp_k(\theta)[1-p_k(\theta)].
$$

The Fisher information supplied by a 2PL item about $\theta$ is
$$
a_k^2p_k(\theta)[1-p_k(\theta)].
$$

Therefore, the information provided by an item increases approximately with $a_k^2$.

When $a_k$ is very large

A large discrimination parameter produces a steep response curve.

The response can strongly distinguish between users whose abilities lie on opposite sides of the item difficulty $b_k$.

If the item is appropriately targeted to the user's ability, the observation contributes substantial information, causing the posterior distribution to become more concentrated or sharper, with a smaller posterior variance.

When $a_k$ is very small

A small discrimination parameter produces a flatter response curve.

The probability of success changes only slowly with $\theta$, so the response contains relatively little information about the user's exact ability.

The posterior changes less after observing the response and generally remains broader, corresponding to greater uncertainty.

The item provides the greatest information around

$$
\theta\approx b_k,
$$

where

$$
p_k(\theta)\approx 0.5.
$$

Thus, both high discrimination and an item difficulty close to the user's current ability are important for reducing posterior uncertainty efficiently.

6. Numerical Implementation Using a Fixed Ability Grid

Because the posterior distribution does not generally have a simple closed-form distribution after applying the logistic likelihood, it can be approximated numerically using a fixed grid.

Step 1: Define a grid

Choose a sufficiently wide range of possible ability values, for example

$$
\theta\in[-4,4].
$$

Represent this range by many equally spaced points:

$$
\theta_1,\theta_2,\ldots,\theta_M.
$$

Step 2: Initialize the prior

Evaluate the standard normal prior at every grid point:
$$
\frac{1}{\sqrt{2\pi}}
\exp\left(-\frac{\theta_j^2}{2}\right).
$$

Normalize numerically so that

$$
\int f^{(0)}(\theta)d\theta=1.
$$

Step 3: Compute the likelihood for the new response

For each grid point, calculate
$$
\frac{1}{1+\exp[-a_k(\theta_j-b_k)]}.
$$

Then calculate
$$
[p_k(\theta_j)]^{y_k}
[1-p_k(\theta_j)]^{1-y_k}.
$$

Step 4: Perform the unnormalized Bayesian update

Multiply the previous posterior by the new likelihood:

L_k(\theta_j)f^{(k-1)}(\theta_j).
$$

Step 5: Normalize computationally

Approximate the normalization constant numerically:
$$
\int \tilde f^{(k)}(\theta)d\theta.
$$

Using a numerical integration method such as the trapezoidal rule,

normalization_constant = np.trapezoid(posterior_unnormalized, theta_grid)

Then normalize:

posterior = posterior_unnormalized / normalization_constant

Hence,
$$
\frac{\tilde f^{(k)}(\theta_j)}
{\int\tilde f^{(k)}(\theta)d\theta}.
$$

This procedure is repeated sequentially after every new item response.

Posterior estimates

The posterior mean is approximated by
$$
E[\Theta\mid y^{(k)}]

\int\theta f^{(k)}(\theta)d\theta.
$$

Numerically,

posterior_mean = np.trapezoid(theta_grid * posterior, theta_grid)

The MAP estimate is
$$
\arg\max_{\theta}f^{(k)}(\theta).
$$

Numerically,

map_estimate = theta_grid[np.argmax(posterior)]
7. Running Simulation for a User with True Ability $\theta_{true}=0.75$

Let

$$
\theta_{true}=0.75
$$

and simulate

$$
n=20
$$

items.

For each item,

$$
b_k\sim\mathcal{N}(0,1)
$$

and

$$
a_k\sim Uniform(0.5,2.0).
$$

The true probability that the user answers item $k$ correctly is
$$
\frac{1}
{1+\exp[-a_k(\theta_{true}-b_k)]}.
$$

Generate

$$
U_k\sim Uniform(0,1).
$$

Then define the simulated response as

$$
y_k=
\begin{cases}
1, & U_k<p_k(\theta_{true}),\
0, & U_k\geq p_k(\theta_{true}).
\end{cases}
$$

The following Python script performs the complete sequential Bayesian simulation and tracks both the posterior mean and MAP estimator.

In [3]:


import numpy as np
import plotly.graph_objects as go

# -------------------------------------------------
# Settings
# -------------------------------------------------

np.random.seed(42)

theta_true = 0.75
n_items = 20

# Fixed ability grid
theta_grid = np.linspace(-4, 4, 4001)

# -------------------------------------------------
# Initial prior: N(0,1)
# -------------------------------------------------

posterior = (1 / np.sqrt(2 * np.pi)) * np.exp(-0.5 * theta_grid**2)

# Normalize numerically
posterior = posterior / np.trapezoid(posterior, theta_grid)

# -------------------------------------------------
# Storage
# -------------------------------------------------

steps = [0]

# At step 0, prior mean and prior MAP are both 0
posterior_means = [
    np.trapezoid(theta_grid * posterior, theta_grid)
]

map_estimates = [
    theta_grid[np.argmax(posterior)]
]

responses = []
difficulties = []
discriminations = []
true_probabilities = []

# -------------------------------------------------
# Sequential Bayesian updating
# -------------------------------------------------

for k in range(1, n_items + 1):

    # Random item parameters
    b_k = np.random.normal(0, 1)
    a_k = np.random.uniform(0.5, 2.0)

    # True probability of a correct response
    p_true = 1 / (
        1 + np.exp(-a_k * (theta_true - b_k))
    )

    # Simulate response using U(0,1)
    u = np.random.uniform(0, 1)

    if u < p_true:
        y_k = 1
    else:
        y_k = 0

    # Probability of success over the entire theta grid
    p_grid = 1 / (
        1 + np.exp(-a_k * (theta_grid - b_k))
    )

    # Likelihood contribution
    likelihood = (
        p_grid**y_k
        * (1 - p_grid)**(1 - y_k)
    )

    # Bayesian update
    posterior_unnormalized = posterior * likelihood

    # Sequential normalization
    normalization_constant = np.trapezoid(
        posterior_unnormalized,
        theta_grid
    )

    posterior = (
        posterior_unnormalized
        / normalization_constant
    )

    # Posterior mean
    posterior_mean = np.trapezoid(
        theta_grid * posterior,
        theta_grid
    )

    # MAP estimate
    map_estimate = theta_grid[
        np.argmax(posterior)
    ]

    # Store results
    steps.append(k)
    posterior_means.append(posterior_mean)
    map_estimates.append(map_estimate)

    responses.append(y_k)
    difficulties.append(b_k)
    discriminations.append(a_k)
    true_probabilities.append(p_true)

# -------------------------------------------------
# Display simulated item information
# -------------------------------------------------

print("True ability =", theta_true)
print()

for k in range(n_items):
    print(
        f"Item {k+1:2d}: "
        f"a={discriminations[k]:.3f}, "
        f"b={difficulties[k]:.3f}, "
        f"P(correct)={true_probabilities[k]:.3f}, "
        f"response={responses[k]}"
    )

# -------------------------------------------------
# Plot estimator progression
# -------------------------------------------------

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_means,
        mode='lines+markers',
        name='Posterior Mean'
    )
)

fig.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimates,
        mode='lines+markers',
        name='MAP Estimate'
    )
)

# True ability reference line
fig.add_hline(
    y=theta_true,
    line_dash='dash',
    annotation_text='True Ability θ = 0.75',
    annotation_position='top left'
)

fig.update_layout(
    title='Sequential Bayesian Estimation of User Ability',
    xaxis_title='Number of Observed Items',
    yaxis_title='Estimated Ability θ',
    template='plotly_white',
    legend_title='Estimator'
)

fig.show()

True ability = 0.75

Item  1: a=1.598, b=0.497, P(correct)=0.600, response=1
Item  2: a=0.734, b=-0.138, P(correct)=0.657, response=1
Item  3: a=0.531, b=1.579, P(correct)=0.392, response=0
Item  4: a=1.749, b=0.767, P(correct)=0.492, response=1
Item  5: a=0.956, b=-0.463, P(correct)=0.761, response=1
Item  6: a=1.148, b=-0.466, P(correct)=0.801, response=1
Item  7: a=0.938, b=-1.013, P(correct)=0.839, response=1
Item  8: a=1.184, b=0.314, P(correct)=0.626, response=0
Item  9: a=1.389, b=0.068, P(correct)=0.721, response=1
Item 10: a=1.411, b=-1.425, P(correct)=0.956, response=1
Item 11: a=1.526, b=-0.601, P(correct)=0.887, response=1
Item 12: a=0.683, b=-0.292, P(correct)=0.671, response=1
Item 13: a=0.968, b=0.823, P(correct)=0.482, response=0
Item 14: a=1.320, b=-1.221, P(correct)=0.931, response=1
Item 15: a=0.633, b=0.738, P(correct)=0.502, response=1
Item 16: a=0.568, b=0.171, P(correct)=0.581, response=1
Item 17: a=1.743, b=-1.479, P(correct)=0.980, response=1
Item 18: a=0.921, 

8. Analysis of Convergence Over Time

At step $0$, no responses have been observed, so the ability estimate is determined entirely by the prior

$$
\Theta\sim\mathcal{N}(0,1).
$$

Hence, both the prior mean and prior MAP estimate are approximately

$$
\hat{\theta}^{(0)}=0.
$$

The true ability is

$$
\theta_{true}=0.75.
$$

As more responses are observed, each response contributes new evidence through the likelihood

$$
[p_k(\theta)]^{y_k}
[1-p_k(\theta)]^{1-y_k}.
$$

The posterior therefore gradually shifts from being dominated by the prior toward being dominated by the observed response data.

In general, as $k$ increases:

The posterior mean and MAP estimates tend to move toward the true ability $\theta_{true}=0.75$.
The estimates may fluctuate, particularly during the early stages, because individual correct or incorrect responses are random.
A surprising response can temporarily move the estimate away from the true value. For example, an incorrect answer to an easy item may reduce the estimated ability, while a correct answer to a difficult item may increase it substantially.
As more informative items are observed, the influence of any single response becomes smaller because the posterior already contains information accumulated from previous responses.
The posterior distribution generally becomes narrower as information accumulates. This represents a reduction in posterior uncertainty.

Therefore, convergence of the posterior mean and MAP toward the true value, together with increasing sharpness of the posterior distribution, indicates that the platform is becoming more confident and more precise in its measurement of the user's latent ability.

However, because only $20$ items are used and the responses are stochastic, the estimates are not guaranteed to equal exactly

$$
\theta_{true}=0.75.
$$

Their convergence also depends on the informativeness of the selected items. Items with high discrimination and difficulties close to the user's true ability generally provide more information and lead to faster reduction in uncertainty.

# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

An e-commerce platform wants to optimize its recommendation engine by dynamically estimating the click-through rate (CTR) of a newly launched advertisement. Since user traffic arrives continuously, the platform updates its belief about the advertisement's performance **one impression at a time** rather than waiting for large batch updates.

Let $\Theta = \theta$ represent the true, hidden conversion rate (probability of a click) of the advertisement, where $\theta \in [0, 1]$.

Let $Y_k$ denote a single user's interaction with the advertisement at time step $k$:

$$Y_k =
\begin{cases}
1, & \text{if the user clicks the advertisement}, \\
0, & \text{if the user does not click the advertisement}.
\end{cases}$$

The platform assumes that conditional on the true conversion rate $\Theta = \theta$, each user interaction is an independent Bernoulli trial:

$$P(Y_k = 1 \mid \Theta = \theta) = \theta$$

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed user interactions** up to the current impression step $k$ (where $1 \le k \le n$).

Before observing any data, the platform assigns a flexible **Beta distribution** as the initial prior over the unknown parameter $\Theta$:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\mathrm{B}(\alpha_0, \beta_0)} \theta^{\alpha_0 - 1} (1 - \theta)^{\beta_0 - 1} \quad \text{implying} \quad \Theta \sim \text{Beta}(\alpha_0, \beta_0)$$

where $\mathrm{B}(\cdot, \cdot)$ is the Beta function acting as the normalizing constant. Under a sequential framework, the posterior distribution at step $k-1$ serves directly as the prior distribution for step $k$.

---

**Tasks**

**1. Structural Probability and Properties**
Plot the probability density function (PDF) of a $\text{Beta}(\alpha, \beta)$ distribution using Plotly for three distinct parameter pairs:

* Uninformative state: $(\alpha=1, \beta=1)$
* Right-skewed state: $(\alpha=2, \beta=8)$
* Left-skewed state: $(\alpha=8, \beta=2)$

Interpret how changing the balance between $\alpha$ and $\beta$ shifts the center of mass of the density function over the domain $[0, 1]$.

**2. Sequential Likelihood and Joint History**

Write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* isolated response $y_k$ at step $k$, given the click probability $\theta$. Following this, express the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

**3. Closed-Form Analytical Updates (Conjugacy)**

Using Bayes' Theorem, derive the recursive algebraic relationship for the posterior density at step $k$, denoted as $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$. Prove analytically that the posterior remains in the Beta family (**Beta-Binomial Conjugacy**) by explicitly writing down the closed-form update parameters $\alpha_k$ and $\beta_k$ as simple arithmetic updates of $\alpha_{k-1}$, $\beta_{k-1}$, and $y_k$. Also compute the **Posterior Mean** of the latent parameter $\Theta$ at time step $k$ (i.e. $\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}]$).


**4. Dynamic Shifting Mechanics**

Explain how an observed click ($y_k = 1$) vs. a non-click ($y_k = 0$) shifts the peak of the running density distribution mathematically. Contrast this analytical framework against non-conjugate setups (such as the 2PL IRT model) where numerical grid integration is strictly required.

**5. Running Point Estimators**

State the exact closed-form equations used to evaluate the following point estimates at step $k$ directly from the updated shape parameters $\alpha_k$ and $\beta_k$:

* **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

**6. Performance Tracking and Convergence Analysis**

Suppose the advertisement's true, hidden click-through rate is $\theta_{\text{true}} = 0.35$. Write a Python script to track the performance of your closed-form sequential estimators over a timeline of $n = 100$ impressions:

* **Initialize State:** Set the base prior parameters to $\alpha_0 = 1, \beta_0 = 1$ (representing uniform initial uncertainty).
* **Simulate Responses:** Dynamically generate user responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against $\theta_{\text{true}}$.
* **Track Estimators:** Loop through each step, update $\alpha_k$ and $\beta_k$ analytically, and store the computed values for $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $100$. Add a static horizontal reference line at $y = 0.35$ representing $\theta_{\text{true}}$.
* **Analysis:** Explain how the distance between your estimators and $\theta_{\text{true}}$ responds as the sampling size $k$ approaches $100$. What does this imply about the accumulation of evidence over time relative to the choice of the initial prior?

# Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

In aerospace and civil engineering, Structural Health Monitoring (SHM) is critical for detecting damage before a catastrophic failure occurs. Consider an aircraft wing or a bridge girder equipped with specialized vibration sensors. Over time, environmental fatigue or dynamic impacts can cause micro-fractures, resulting in a reduction of the component's mechanical stiffness.

Let $\Theta = \theta$ represent the structural **remaining stiffness efficiency factor**, where $\theta$ is physically bounded to the interval:

$$\theta \in (0, 1]$$

* $\theta = 1.0$ indicates a perfectly pristine, undamaged structural component.
* $\theta \to 0$ signifies critical degradation or severe structural cracking.

Let $K_{\text{nominal}}$ be the known, baseline stiffness of the structural component when it is entirely healthy. At each sequential inspection time step $k$ (where $k = 1, 2, \dots, n$), a sensor collects a noisy experimental stiffness measurement $y_k$.

Engineers model the degradation physics via a non-linear relationship with multiplicative log-normal measurement noise to prevent non-physical negative values:

$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \qquad \epsilon_k \sim \mathscr{N}(0, \sigma^2)$$

where $\sigma$ is the standard deviation of the sensor noise in log-space.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running history vector of observed sensor readings** up to the current inspection milestone. Before deploying the sensors, engineers utilize an initial prior distribution $f_{\Theta}^{(0)}(\theta)$ over the domain $(0, 1]$ based on historical manufacturing specifications. As the sensor stream arrives, the posterior distribution calculated at step $k-1$ serves directly as the prior distribution for step $k$.

---

### **Tasks**

#### **1. Prior Belief Boundaries**

Before data collection begins, engineers assume the component is highly likely to be healthy, modeling this using a bounded Beta distribution as the initial prior: $\Theta \sim \text{Beta}(8, 1.5)$.

* Plot this initial prior density function using Plotly over the restricted physical domain $\theta \in [0.01, 1.0]$.
* Calculate the expected prior stiffness efficiency $\mathbb{E}[\Theta^{(0)}]$ analytically. Explain why this specific distribution serves as an appropriate initial prior for an engineering component assumed to be healthy.

#### **2. Structural Likelihood Formulation**

Using the change of variables or properties of the log-normal distribution, write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* continuous sensor measurement $y_k$ at inspection step $k$, given the true stiffness factor $\theta$. Following this, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

#### **3. Mathematical Formulation of the Non-Conjugate Grid Update**

Explain why an exact closed-form analytical solution for the posterior density $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ does not exist when combining a Beta prior with this log-normal structural likelihood. Write down the recursive relationship for the posterior density at step $k$ up to a proportionality constant.

#### **4. Running Point Estimates**

Because a closed-form formula is unavailable, we must define point estimators through numerical integration. Write down the definite integral equations over the bounded domain $(0, 1]$ required to compute:

* The **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* The **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

#### **5. Algorithmic Grid Approximation and Normalization**

Describe the step-by-step numerical procedure to maintain this distribution on a discrete grid of $\theta$-values. Explicitly state how you would handle the boundary limits computationally and how you would perform the sequential normalization step using the trapezoidal rule after a new sensor reading $y_k$ is observed.

#### **6. Performance Tracking and Degradation Convergence Analysis**

Suppose an impact occurs, and the true, hidden remaining stiffness drops to $\theta_{\text{true}} = 0.68$. Write a Python script using Plotly to simulate an engineered monitoring timeline across $n = 15$ continuous sensor measurements ($K_{\text{nominal}} = 50.0 \text{ kN/mm}$, $\sigma = 0.15$):

* **Simulate Sensor Stream:** Programmatically generate noisy sensor readings $y_k$ by drawing random values from the underlying log-normal physics model centered at $\theta_{\text{true}}$.
* **Track Estimators:** Loop sequentially through each step. At each step, update the unnormalized grid, normalize it via `np.trapezoid`, and compute both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize Curves & Timeline:** Generate two plots:
1. A plot showing the progression of the full posterior density curves at milestones $k \in \{0, 1, 2, 5, 10, 15\}$.
2. A line chart tracking the convergence of both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$ from step $0$ to $15$ against a horizontal reference line at $\theta_{\text{true}} = 0.68$.


* **Analysis:** Evaluate the behavior of the distribution. How many sensor readings did it take for the system to overcome the initially optimistic "healthy" prior and confidently isolate the 68% damage state? What does the narrowing of the density curves imply about structural safety thresholds?

#Answers

1. Prior Belief Boundaries

Before receiving any sensor measurements, assume

$$
\Theta\sim\text{Beta}(8,1.5).
$$

The Beta density is
$$
\frac{1}{B(8,1.5)}
\theta^{8-1}(1-\theta)^{1.5-1},
\qquad 0<\theta<1,
$$

where $B(\alpha,\beta)$ is the Beta function.

In [5]:
#Plot of the Initial Prior

import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# Beta prior parameters
alpha = 8.0
beta_param = 1.5

# Restricted physical grid

theta_grid = np.linspace(0.01, 1.0, 4001)

# Evaluate Beta prior
prior = beta.pdf(theta_grid, alpha, beta_param)

# Normalize numerically over the computational grid
prior = prior / np.trapezoid(prior, theta_grid)

# Plot
fig = go.Figure()
fig.add_trace( go.Scatter(
    x=theta_grid,
    y=prior,
    mode='lines',
    name='Beta(8, 1.5) Prior' ) )

fig.update_layout(
    title='Initial Prior Distribution of Structural Stiffness Efficiency',
    xaxis_title='Remaining Stiffness Efficiency θ',
    yaxis_title='Prior Density',
    template='plotly_white' )

fig.show()

Expected Prior Stiffness Efficiency

For a Beta distribution,
$$
\frac{\alpha}{\alpha+\beta}.
$$

Therefore,
$$
\frac{8}{8+1.5}
$$
$$
\frac{8}{9.5}
\approx0.8421.
$$

Hence,

$$
\boxed{E[\Theta^{(0)}]\approx0.842}
$$

This means that, before collecting any measurements, engineers expect the component to retain approximately 84.2% of its nominal stiffness.

The distribution $\text{Beta}(8,1.5)$ is appropriate because:

The Beta distribution is naturally restricted to the physical interval $(0,1)$.
Most prior probability is concentrated toward large values of $\theta$, representing the engineering belief that a newly monitored component is likely to be healthy.
It still allows probability for lower stiffness values, representing uncertainty about manufacturing defects, fatigue, or previously undetected damage.
It does not assign probability to physically impossible values such as $\theta<0$ or $\theta>1$.

Thus, it provides an appropriate bounded and optimistic prior belief for a structure initially assumed to be in good condition.

2. Structural Likelihood Formulation

The measurement model is
$$
\theta K_{\text{nominal}}e^{\epsilon_k},
$$

with

$$
\epsilon_k\sim\mathcal{N}(0,\sigma^2).
$$

Taking the natural logarithm,
$$
\ln(\theta K_{\text{nominal}})
+
\epsilon_k.
$$

Therefore,

$$
\ln y_k\mid\theta
\sim
\mathcal{N}
\left(
\ln(\theta K_{\text{nominal}}),
\sigma^2
\right).
$$

Hence,

$$
Y_k\mid\Theta=\theta
\sim
\text{LogNormal}
\left(
\ln(\theta K_{\text{nominal}}),
\sigma^2
\right).
$$

The likelihood contribution of one measurement $y_k$ is therefore
$$
\frac{1}
{y_k\sigma\sqrt{2\pi}}
\exp
\left[
-\frac{
\left(
\ln y_k-\ln(\theta K_{\text{nominal}})
\right)^2
}
{2\sigma^2}
\right],
$$

for

$$
y_k>0,\qquad 0<\theta\leq1.
$$

Assuming the measurements are conditionally independent given $\theta$, the joint likelihood of the running measurement history
$$
(y_1,y_2,\ldots,y_k)
$$

is
$$
\prod_{i=1}^{k}
L(y_i\mid\theta).
$$

Therefore,
$$
\prod_{i=1}^{k}
\left[
\frac{1}
{y_i\sigma\sqrt{2\pi}}
\exp
\left(
-\frac{
[\ln y_i-\ln(\theta K_{\text{nominal}})]^2
}
{2\sigma^2}
\right)
\right].
$$

3. Mathematical Formulation of the Non-Conjugate Grid Update

The initial prior is a Beta distribution:

$$
f_\Theta^{(0)}(\theta)
\propto
\theta^{\alpha-1}(1-\theta)^{\beta-1}.
$$

However, the likelihood contains the nonlinear logarithmic term

$$
\exp
\left[
-\frac{
(\ln y_k-\ln(\theta K_{\text{nominal}}))^2
}
{2\sigma^2}
\right].
$$

Multiplying the Beta prior by this log-normal likelihood does not produce another Beta distribution or another standard probability distribution with simple updated parameters.

Therefore, the Beta prior is not conjugate to this log-normal likelihood.

As a result, there is no convenient closed-form posterior distribution, and numerical methods such as grid approximation are required.

At step $k-1$, suppose the current posterior is

$$
f_{\Theta\mid Y^{(k-1)}}
\left(
\theta\mid y^{(k-1)}
\right).
$$

After observing the new sensor measurement $y_k$, Bayes' theorem gives

$$
f_{\Theta\mid Y^{(k)}}
\left(
\theta\mid y^{(k)}
\right)
\propto
L(y_k\mid\theta)
f_{\Theta\mid Y^{(k-1)}}
\left(
\theta\mid y^{(k-1)}
\right).
$$

Substituting the log-normal likelihood,

$$
f_{\Theta\mid Y^{(k)}}
\left(
\theta\mid y^{(k)}
\right)
\propto
\frac{1}
{y_k\sigma\sqrt{2\pi}}
\exp
\left[
-\frac{
(\ln y_k-\ln(\theta K_{\text{nominal}}))^2
}
{2\sigma^2}
\right]
f_{\Theta\mid Y^{(k-1)}}
\left(
\theta\mid y^{(k-1)}
\right).
$$

Thus, the posterior obtained after step $k$ becomes the prior for step $k+1$.

The normalized recursive form is
$$
\frac{
L(y_k\mid\theta)
f_{\Theta\mid Y^{(k-1)}}(\theta\mid y^{(k-1)})
}{
\displaystyle
\int_0^1
L(y_k\mid u)
f_{\Theta\mid Y^{(k-1)}}(u\mid y^{(k-1)})
,du
}.
$$

4. Running Point Estimates
Running Posterior Mean

The Bayesian posterior mean at step $k$ is
$$
E[\Theta\mid y^{(k)}].
$$

Therefore,
$$
\int_0^1
\theta
f_{\Theta\mid Y^{(k)}}
(\theta\mid y^{(k)})
,d\theta
}
$$

If an unnormalized posterior $\tilde f^{(k)}(\theta)$ is being used, the equivalent expression is
$$
\frac{
\displaystyle
\int_0^1
\theta\tilde f^{(k)}(\theta),d\theta
}{
\displaystyle
\int_0^1
\tilde f^{(k)}(\theta),d\theta
}

$$

Running Maximum A Posteriori Estimate

The MAP estimator is the value of $\theta$ where the posterior density reaches its maximum:
$$
\underset{0<\theta\leq1}{\arg\max}
;
f_{\Theta\mid Y^{(k)}}
(\theta\mid y^{(k)})

$$

Because multiplying a density by a positive normalization constant does not change the location of its maximum,
$$
\underset{0<\theta\leq1}{\arg\max}
;
\tilde f^{(k)}(\theta).
$$

Therefore, the posterior mean uses numerical integration, whereas the MAP estimate is obtained by finding the grid point having the largest posterior density.

5. Algorithmic Grid Approximation and Normalization

Because an analytical posterior distribution is unavailable, the posterior can be maintained numerically on a fixed grid.

Step 1: Construct the Bounded Grid

The physical domain is

$$
0<\theta\leq1.
$$

Since the likelihood contains

$$
\ln(\theta K_{\text{nominal}}),
$$

we cannot evaluate the likelihood exactly at $\theta=0$ because $\ln(0)$ is undefined.

Therefore, choose a small positive lower boundary such as

$$
\theta_{\min}=0.01
$$

and define

$$
\theta_j\in[0.01,1.0].
$$

For example:

theta_grid = np.linspace(0.01, 1.0, 4001)

Using many grid points provides a good numerical approximation while respecting the physical upper limit

$$
\theta\leq1.
$$

Step 2: Initialize the Prior

Evaluate
$$
\text{Beta}(\theta_j;8,1.5)
$$

at every grid point.

Then numerically normalize it:

$$
f^{(0)}(\theta_j)
\leftarrow
\frac{
f^{(0)}(\theta_j)
}{
\displaystyle
\int_{0.01}^{1}
f^{(0)}(\theta),d\theta
}.
$$

Computationally:

posterior = beta.pdf(theta_grid, 8, 1.5)

posterior = posterior / np.trapezoid(
    posterior,
    theta_grid
)
Step 3: Evaluate the New Likelihood

When measurement $y_k$ arrives, evaluate
$$
\frac{1}
{y_k\sigma\sqrt{2\pi}}
\exp
\left[
-\frac{
(\ln y_k-\ln(\theta_jK_{\text{nominal}}))^2
}
{2\sigma^2}
\right]
$$

at every grid point.

Step 4: Perform the Bayesian Multiplication

Compute the unnormalized posterior:
$$
L_k(\theta_j)
f^{(k-1)}(\theta_j).
$$

Computationally:

posterior_unnormalized = posterior * likelihood
Step 5: Sequential Normalization Using the Trapezoidal Rule

Calculate the normalization constant
$$
\int_{0.01}^{1}
\tilde f^{(k)}(\theta),d\theta.
$$

Using the trapezoidal rule:

normalization_constant = np.trapezoid(
    posterior_unnormalized,
    theta_grid
)

Then normalize:

posterior = (
    posterior_unnormalized
    / normalization_constant
)

Hence,
$$
\frac{
\tilde f^{(k)}(\theta_j)
}{
Z_k
}.
$$

This guarantees approximately




$$

Step 6: Calculate the Running Estimators

Posterior mean:

posterior_mean = np.trapezoid(
    theta_grid * posterior,
    theta_grid
)

MAP estimate:

map_estimate = theta_grid[
    np.argmax(posterior)
]

The updated posterior is then retained and used as the prior for the next sensor observation.

6. Performance Tracking and Degradation Convergence Analysis

Assume the actual remaining stiffness efficiency after an impact is

$$
\theta_{\text{true}}=0.68.
$$

Therefore, the component retains approximately 68% of its nominal stiffness, corresponding to approximately 32% stiffness loss relative to the pristine condition.

Let

$$
K_{\text{nominal}}=50.0\text{ kN/mm},
$$

$$
\sigma=0.15,
$$

and

$$
n=15.
$$

The true noise-free stiffness after damage is
$$
0.68(50)

34.0\text{ kN/mm}.
$$

The actual sensor measurements fluctuate around this value according to the log-normal noise model.

In [6]:
#Complete Python Simulation
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# --------------------------------------------------
# Reproducibility
# --------------------------------------------------

np.random.seed(42)

# --------------------------------------------------
# Structural parameters
# --------------------------------------------------

theta_true = 0.68
K_nominal = 50.0       # kN/mm
sigma = 0.15
n_measurements = 15

# --------------------------------------------------
# Fixed bounded theta grid
# --------------------------------------------------

theta_grid = np.linspace(0.01, 1.0, 4001)

# --------------------------------------------------
# Initial Beta(8, 1.5) prior
# --------------------------------------------------

alpha = 8.0
beta_param = 1.5

posterior = beta.pdf(
    theta_grid,
    alpha,
    beta_param
)

# Normalize initial prior numerically
posterior = posterior / np.trapezoid(
    posterior,
    theta_grid
)

# --------------------------------------------------
# Storage
# --------------------------------------------------

steps = [0]

posterior_means = [
    np.trapezoid(
        theta_grid * posterior,
        theta_grid
    )
]

map_estimates = [
    theta_grid[np.argmax(posterior)]
]

sensor_readings = []

# Store posterior curves at selected milestones
milestones = [0, 1, 2, 5, 10, 15]

posterior_curves = {
    0: posterior.copy()
}

# Store 95% credible intervals
credible_intervals = []

# --------------------------------------------------
# Helper function for a numerical credible interval
# --------------------------------------------------

def credible_interval(theta_grid, posterior, level=0.95):

    # Numerical cumulative integral using trapezoids
    dx = np.diff(theta_grid)

    cumulative = np.concatenate([
        [0.0],
        np.cumsum(
            0.5
            * (posterior[:-1] + posterior[1:])
            * dx
        )
    ])

    # Ensure final value is exactly normalized
    cumulative = cumulative / cumulative[-1]

    alpha_tail = (1 - level) / 2

    lower = np.interp(
        alpha_tail,
        cumulative,
        theta_grid
    )

    upper = np.interp(
        1 - alpha_tail,
        cumulative,
        theta_grid
    )

    return lower, upper

# --------------------------------------------------
# Sequential sensor measurements
# --------------------------------------------------

for k in range(1, n_measurements + 1):

    # Simulate multiplicative Gaussian noise
    epsilon_k = np.random.normal(
        loc=0.0,
        scale=sigma
    )

    # Generate log-normal stiffness measurement
    y_k = (
        theta_true
        * K_nominal
        * np.exp(epsilon_k)
    )

    sensor_readings.append(y_k)

    # --------------------------------------------------
    # Likelihood over theta grid
    # --------------------------------------------------

    likelihood = (
        1
        /
        (
            y_k
            * sigma
            * np.sqrt(2 * np.pi)
        )
    ) * np.exp(
        -
        (
            np.log(y_k)
            -
            np.log(theta_grid * K_nominal)
        )**2
        /
        (2 * sigma**2)
    )

    # --------------------------------------------------
    # Bayesian update
    # --------------------------------------------------

    posterior_unnormalized = (
        posterior * likelihood
    )

    # Sequential normalization using trapezoidal rule
    normalization_constant = np.trapezoid(
        posterior_unnormalized,
        theta_grid
    )

    posterior = (
        posterior_unnormalized
        /
        normalization_constant
    )

    # --------------------------------------------------
    # Running posterior mean
    # --------------------------------------------------

    posterior_mean = np.trapezoid(
        theta_grid * posterior,
        theta_grid
    )

    # --------------------------------------------------
    # Running MAP estimate
    # --------------------------------------------------

    map_estimate = theta_grid[
        np.argmax(posterior)
    ]

    # Store estimates
    steps.append(k)
    posterior_means.append(posterior_mean)
    map_estimates.append(map_estimate)

    # Store selected posterior curves
    if k in milestones:
        posterior_curves[k] = posterior.copy()

    # Calculate 95% credible interval
    lower, upper = credible_interval(
        theta_grid,
        posterior
    )

    credible_intervals.append(
        (lower, upper)
    )

# --------------------------------------------------
# Display numerical results
# --------------------------------------------------

print(
    f"True remaining stiffness = "
    f"{theta_true:.3f}"
)

print(
    f"Initial prior mean = "
    f"{posterior_means[0]:.3f}"
)

print()

for k in range(1, n_measurements + 1):

    lower, upper = credible_intervals[k - 1]

    print(
        f"Step {k:2d}: "
        f"Measurement = {sensor_readings[k-1]:6.2f} kN/mm, "
        f"Mean = {posterior_means[k]:.3f}, "
        f"MAP = {map_estimates[k]:.3f}, "
        f"95% CI = "
        f"[{lower:.3f}, {upper:.3f}]"
    )

True remaining stiffness = 0.680
Initial prior mean = 0.842

Step  1: Measurement =  36.63 kN/mm, Mean = 0.808, MAP = 0.816, 95% CI = [0.625, 0.968]
Step  2: Measurement =  33.30 kN/mm, Mean = 0.752, MAP = 0.744, 95% CI = [0.612, 0.906]
Step  3: Measurement =  37.47 kN/mm, Mean = 0.752, MAP = 0.745, 95% CI = [0.635, 0.882]
Step  4: Measurement =  42.73 kN/mm, Mean = 0.776, MAP = 0.770, 95% CI = [0.670, 0.891]
Step  5: Measurement =  32.83 kN/mm, Mean = 0.751, MAP = 0.747, 95% CI = [0.658, 0.853]
Step  6: Measurement =  32.83 kN/mm, Mean = 0.735, MAP = 0.731, 95% CI = [0.651, 0.826]
Step  7: Measurement =  43.09 kN/mm, Mean = 0.752, MAP = 0.748, 95% CI = [0.672, 0.837]
Step  8: Measurement =  38.15 kN/mm, Mean = 0.753, MAP = 0.750, 95% CI = [0.678, 0.833]
Step  9: Measurement =  31.69 kN/mm, Mean = 0.739, MAP = 0.736, 95% CI = [0.670, 0.813]
Step 10: Measurement =  36.88 kN/mm, Mean = 0.739, MAP = 0.736, 95% CI = [0.673, 0.809]
Step 11: Measurement =  31.72 kN/mm, Mean = 0.729, MAP = 0.

In [8]:
#Plot 1: Evolution of the Full Posterior Density
fig1 = go.Figure()

for k in milestones:

    fig1.add_trace(
        go.Scatter(
            x=theta_grid,
            y=posterior_curves[k],
            mode='lines',
            name=f'k = {k}'
        )
    )

fig1.add_vline(
    x=theta_true,
    line_dash='dash',
    annotation_text='True θ = 0.68',
    annotation_position='top'
)

fig1.update_layout(
    title='Evolution of Posterior Density for Remaining Structural Stiffness',
    xaxis_title='Remaining Stiffness Efficiency θ',
    yaxis_title='Posterior Density',
    template='plotly_white',
    legend_title='Inspection Step'
)

fig1.show()

#Plot 2: Convergence of Posterior Mean and MAP
fig2 = go.Figure()

fig2.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_means,
        mode='lines+markers',
        name='Posterior Mean'
    )
)

fig2.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimates,
        mode='lines+markers',
        name='MAP Estimate'
    )
)

fig2.add_hline(
    y=theta_true,
    line_dash='dash',
    annotation_text='True θ = 0.68',
    annotation_position='top left'
)

fig2.update_layout(
    title='Sequential Convergence of Structural Stiffness Estimates',
    xaxis_title='Number of Sensor Measurements',
    yaxis_title='Estimated Remaining Stiffness Efficiency θ',
    template='plotly_white',
    legend_title='Estimator'
)

fig2.show()

Determining When the Healthy Prior Has Been Overcome

The exact number of measurements required to "confidently" detect degradation depends on the random sensor measurements and on the engineering definition of confidence.

A reproducible criterion can therefore be defined.

For example, we may say that the initially optimistic healthy prior has been overcome when:

The posterior mean is within $0.05$ of the true stiffness value:

$$
|\hat{\theta}_{Bayes}^{(k)}-0.68|<0.05,
$$

and

The upper boundary of the 95% credible interval is below $0.80$:

$$
\theta_{97.5%}^{(k)}<0.80
$$

The following code determines the first step satisfying both conditions.

In [10]:
confidence_step = None
for k in range(1, n_measurements + 1):
  lower, upper = credible_intervals[k - 1]

  mean_close = (
      abs(
          posterior_means[k] - theta_true ) < 0.05 )

  clearly_degraded = upper < 0.80

  if mean_close and clearly_degraded:
    confidence_step = k
    break

if confidence_step is not None:
  print(
      "Confidence criterion first satisfied "
      f"after {confidence_step} measurements." )
else:
  print( "Confidence criterion was not satisfied " "within the 15 measurements." )

Confidence criterion first satisfied after 11 measurements.


With

np.random.seed(42)

this simulation first satisfies the stated criterion at approximately

$$
\boxed{k=11}
$$

sensor measurements.

Thus, in this particular reproducible simulation, approximately 11 measurements are required for the accumulated sensor evidence to clearly overcome the initially optimistic healthy prior and identify a state close to

$$
\theta_{\text{true}}=0.68.
$$

Because the observations contain random noise, this exact number will change if a different random seed or different confidence criterion is used.

Analysis of the Posterior Evolution

Initially, the prior distribution

$$
\Theta\sim\text{Beta}(8,1.5)
$$

is strongly biased toward relatively healthy stiffness values.

Its expected value is

$$
E[\Theta^{(0)}]\approx0.842.
$$

Therefore, before any sensor measurements are collected, the monitoring system expects approximately 84% remaining stiffness.

However, the true post-impact state is

$$
\theta_{\text{true}}=0.68.
$$

The corresponding true noise-free stiffness is

0.68(50)

34\text{ kN/mm}.
$$

As measurements near this degraded stiffness level are observed, their likelihood is greater for values of $\theta$ near $0.68$ than for values near the initially expected healthy state.

The sequential update

$$
f^{(k)}(\theta)
\propto
L(y_k\mid\theta)
f^{(k-1)}(\theta)
$$

therefore gradually shifts probability mass away from the optimistic prior region and toward the actual degraded state.

For the reproducible simulation using random seed 42, the posterior mean and MAP progressively approach the true state. By step 15 they are approximately

$$
\hat{\theta}_{Bayes}^{(15)}
\approx0.689
$$

and

$$
\hat{\theta}_{MAP}^{(15)}
\approx0.687.
$$

These values are close to

$$
\theta_{\text{true}}=0.68.
$$

The 95% credible interval at step 15 is approximately

$$
[0.638,;0.742].
$$

Therefore, the data have shifted the system substantially away from its original healthy expectation and concentrated the probability around the degraded stiffness region.

Interpretation of Posterior Narrowing for Structural Safety

The narrowing of the posterior density curves represents a reduction in uncertainty about the remaining stiffness.

At the beginning, the posterior is broad because the estimate depends mainly on prior engineering knowledge.

After additional measurements,

$$
f_{\Theta\mid Y^{(k)}}(\theta\mid y^{(k)})
$$

becomes more concentrated because more sensor evidence has been accumulated.

A narrow posterior around

$$
\theta\approx0.68
$$

means that the monitoring system is increasingly confident that the component retains approximately 68% of its nominal stiffness.

This is particularly important when structural safety decisions are based on a threshold.

For example, suppose maintenance is required when

$$
\theta<0.75.
$$

It is more informative to calculate

$$
P(\Theta<0.75\mid y^{(k)})
$$

than to rely only on a single point estimate.

If most posterior probability lies below the safety threshold, engineers have strong probabilistic evidence that intervention is required.

The posterior probability below a safety threshold $\theta_{\text{safe}}$ is

\int_0^{\theta_{\text{safe}}}
f_{\Theta\mid Y^{(k)}}
(\theta\mid y^{(k)})
,d\theta.
$$

For example, computationally:

theta_safe = 0.75

mask = theta_grid <= theta_safe

prob_below_safe = np.trapezoid(
    posterior[mask],
    theta_grid[mask]
)

print(
    f"P(Theta < {theta_safe}) = "
    f"{prob_below_safe:.4f}"
)

Therefore, narrowing of the posterior has an important engineering interpretation:

A wide posterior means substantial uncertainty remains about structural integrity.
A narrow posterior above a safety threshold provides stronger evidence that the structure remains acceptable.
A narrow posterior below a safety threshold provides strong evidence of degradation and may trigger inspection, repair, load restrictions, or component replacement.

Hence, sequential Bayesian updating provides more than a single damage estimate. It also quantifies the confidence and uncertainty associated with structural safety decisions.

Final Conclusion

The bounded Bayesian grid method successfully combines an engineering prior with sequential noisy stiffness measurements.

The initial prior

$$
\Theta\sim\text{Beta}(8,1.5)
$$

represents an initially healthy component with expected remaining stiffness

$$
E[\Theta]\approx0.842.
$$

Following an impact that reduces the true remaining stiffness to

$$
\theta_{\text{true}}=0.68,
$$

successive measurements gradually overcome this optimistic prior.

For the reproducible simulation with seed 42, the defined confidence criterion is reached after approximately

$$
\boxed{11\text{ sensor measurements}}.
$$

By 15 measurements, the posterior mean and MAP estimates are both close to the true value of $0.68$, while the posterior distribution becomes substantially narrower.

This narrowing indicates increasing confidence in the estimated degradation state and allows engineers to make probabilistic decisions regarding structural safety thresholds rather than relying only on uncertain individual sensor measurements.

# Q. Gaussian Mixture Clustering as Conditional Updating

Consider a dataset
$$
x_1,x_2,\dots,x_n\in\mathbb R^d.
$$
We wish to cluster these observations into $K$ groups. Instead of assigning each point deterministically to a cluster at the beginning, we introduce a latent random variable
$$
C_i\in{1,\dots,K},
$$
where $C_i=k$ means that the observation $x_i$ belongs to cluster $k$.
Let the prior probability of cluster membership be
$$
P(C_i=k)=\phi_k,
$$
where
$$
\phi_k\ge 0,
\qquad
\sum_{k=1}^K \phi_k=1.
$$

Conditional on $C_i=k$, assume that the observation $X_i$ is generated from a multivariate Gaussian distribution:
$$
X_i\mid C_i=k
\sim
\mathscr N(\mu_k,\Sigma_k),
$$
where
$$
\mu_k\in\mathbb R^d,
\qquad
\Sigma_k\in\mathbb R^{d\times d}
$$
are the mean vector and covariance matrix of cluster $k$.

The model parameters
$$
\phi_k,\mu_k,\Sigma_k,
\qquad k=1,\dots,K,
$$
are assumed to be fixed but unknown.

---

1. Deriving the Marginal Density:
Using the law of total probability, show that the marginal density of $X_i$ is
$$
p(x_i)=\sum_{k=1}^K
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k).
$$
Explain why this density is called a Gaussian mixture density.

---

2. Deriving the Posterior Cluster Probability:
For a fixed observation $x_i$, use Bayes' rule to derive
$$
P(C_i=k\mid X_i=x_i)=\frac{
P(X_i=x_i\mid C_i=k)P(C_i=k)
}{
\sum_{j=1}^K P(X_i=x_i\mid C_i=j)P(C_i=j)
}.
$$
Then substitute the Gaussian model and the cluster prior to obtain
$$
P(C_i=k\mid X_i=x_i)=\frac{
\phi_k\mathscr N(x_i\mid \mu_k,\Sigma_k)
}{
\sum_{j=1}^K
\phi_j\mathscr N(x_i\mid \mu_j,\Sigma_j)
}.
$$
This quantity is called the responsibility of cluster $k$ for data point $x_i$, and is denoted by
$$
\gamma_{ik}=P(C_i=k\mid X_i=x_i).
$$
Explain why $\gamma_{ik}$ may be interpreted as a posterior probability of cluster membership.

---

3. One-Hot Encoding of the Latent Cluster Variable:
Now define a one-hot encoded latent random vector
$$
Z_i=
\begin{bmatrix}
Z_{i1}\\
Z_{i2}\\
\vdots\\
Z_{iK}
\end{bmatrix},
$$
where
$$
Z_{ik}=\begin{cases}
1, & \text{if } C_i=k,\\
0, & \text{otherwise}.
\end{cases}
$$
Show that
$$
\mathbb E[Z_{ik}\mid X_i=x_i]=P(C_i=k\mid X_i=x_i).
$$
Hence show that
$$
\mathbb E[Z_i\mid X_i=x_i]=\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}.
$$
Conclude that the soft cluster assignment in a Gaussian mixture model is precisely the conditional expectation
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

---

4. From Soft Assignment to Hard Clustering:
The vector
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
gives a soft assignment of $x_i$ to all clusters. A hard cluster assignment can be obtained by choosing the cluster with the largest posterior probability:
$$
\widehat C_i=\operatorname{arg\,max}_{1\le k\le K}
\gamma_{ik}.
$$
Explain the difference between soft clustering and hard clustering in this context.

---

5. Conditional Expectation of the Observation Given the Cluster:
Show that
$$
\mathbb E[X_i\mid C_i=k]=\mu_k.
$$
Explain why $\mu_k$ can be interpreted as the center of cluster $k$.
Then compare the two conditional expectations
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
and
$$
\mathbb E[X_i\mid C_i=k].
$$
Explain why the first gives the soft cluster membership of an observed point, while the second gives the mean location of a cluster.

---

6. The Complete-Data Likelihood
If the latent labels $z_i$ were known, the complete-data likelihood would be
$$
p(x_1,\dots,x_n,z_1,\dots,z_n)=\prod_{i=1}^n
\prod_{k=1}^K
\left[
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k)
\right]^{z_{ik}}.
$$
Take the logarithm and show that the complete-data log-likelihood is
$$
\ell_c=\sum_{i=1}^n
\sum_{k=1}^K
z_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why this expression would be easy to maximize if the values of $z_{ik}$ were known.

---

7. The EM Interpretation:
In practice, the latent variables $Z_i$ are not observed. The EM algorithm replaces the unknown indicators $z_{ik}$ by their conditional expectations given the observed data and current parameter estimates:
$$
z_{ik}
\quad\leadsto\quad
\mathbb E[Z_{ik}\mid X_i=x_i].
$$
That is,
$$
z_{ik}
\quad\leadsto\quad
\gamma_{ik}.
$$
This is the E-step of the EM algorithm.
Using this idea, write the expected complete-data log-likelihood:
$$
Q=\sum_{i=1}^n
\sum_{k=1}^K
\gamma_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why the E-step can be interpreted as a conditional update of cluster membership probabilities.

---

8. Parameter Updates:
By maximizing $Q$ with respect to the model parameters, derive the standard GMM updates:
$$
N_k=\sum_{i=1}^n \gamma_{ik},
$$
$$
\phi_k^{\text{new}}=\frac{N_k}{n},
$$
$$
\mu_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}x_i,
$$
and
$$
\Sigma_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}
(x_i-\mu_k^{\text{new}})
(x_i-\mu_k^{\text{new}})^T.
$$
Explain how the responsibility $\gamma_{ik}$ acts as a fractional membership weight of observation $x_i$ in cluster $k$.

---

9. Interpretation:
Write a short paragraph explaining why GMM clustering can be viewed as a repeated process of conditional updating.
Your answer should mention the following points:

* The mixture weight $\phi_k$ is the prior probability of cluster $k$.
* The Gaussian density $\mathscr N(x_i\mid \mu_k,\Sigma_k)$ measures how compatible $x_i$ is with cluster $k$.
* The responsibility $\gamma_{ik}$ is the posterior probability of cluster $k$ after observing $x_i$.
* The soft assignment vector is
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

* The M-step updates the cluster parameters using these posterior membership probabilities as weights.
Conclude that Gaussian mixture clustering is probabilistic clustering based on conditional expectations of latent cluster membership variables.

---

Here is a perfectly tailored question that you can add as the final part (**Part 10**) of your assignment notebook to bridge your theoretical derivations with your code implementation:

---

10. Computational Simulation and Out-of-Sample Validation

Using the theoretical framework established in the previous parts, write a Python class named `GMMFinancialSegmenter` that implements a two-dimensional Gaussian Mixture Model (GMM) using `scikit-learn` and visualizes the results interactively using `Plotly`. Your implementation should fulfill the following criteria:

* **Data Splitting and Scaling:** Accept a dataset containing two continuous features (e.g., mimicking financial behaviors like `PURCHASES` and `CREDIT_LIMIT`), standardize the features to handle variance scaling, and split the data into an 80% training set and a 20% validation/test set.
* **EM Execution:** Fit a GMM with $K=3$ components on the training data using the Expectation-Maximization (EM) algorithm, printing whether the model successfully converged and the number of iterations required.
* **Out-of-Sample Performance:** Compute and output the average log-likelihood score over the unseen test set to validate how well the learned density functions generalize to new data.
* **Interactive Visualizations:** Implement methods to generate three distinct Plotly figures:
1. An empirical **2D Density Heatmap** of the raw training data with marginal distributions to inspect its underlying multimodal structure.
2. A **Training Assignment Plot** that overlays the training data points on top of a continuous contour map showing the maximum posterior responsibilities ($\gamma_{ik}$) computed across a fine coordinate grid.
3. A **Test Assignment Plot** that replicates the contour boundary visualization but overlays out-of-sample test data points to expose the physical regions of cluster ambiguity.



Briefly evaluate the resulting plots. Explain how the continuous background contour map visually demonstrates the soft assignment expectation vector $\mathbb{E}[Z_i \mid X_i = x_{\text{grid}}]$ that you proved analytically in Part 3.

Use the dataset

https://www.kaggle.com/datasets/arjunbhasin2013/ccdata

#Answers

Gaussian Mixture Clustering as Conditional Updating

Consider observations

$$
x_1,x_2,\ldots,x_n\in\mathbb{R}^d
$$

that we wish to divide into $K$ clusters.

Introduce a latent cluster variable

$$
C_i\in{1,\ldots,K},
$$

where

$$
C_i=k
$$

means that observation $x_i$ belongs to cluster $k$.

The prior probability of cluster membership is

$$
P(C_i=k)=\phi_k,
$$

where

$$
\phi_k\geq0,
\qquad
\sum_{k=1}^{K}\phi_k=1.
$$

Conditional on belonging to cluster $k$, assume

$$
X_i\mid C_i=k
\sim
\mathcal{N}(\mu_k,\Sigma_k),
$$

where

$$
\mu_k\in\mathbb{R}^d
$$

is the mean of cluster $k$ and

$$
\Sigma_k\in\mathbb{R}^{d\times d}
$$

is its covariance matrix.

1. Deriving the Marginal Density

Using the law of total probability, the density of $X_i$ can be obtained by summing over all possible latent clusters:

\sum_{k=1}^{K}
p(x_i,C_i=k).
$$

Using

p(x_i\mid C_i=k)P(C_i=k),
$$

we obtain

\sum_{k=1}^{K}
P(C_i=k)
p(x_i\mid C_i=k).
$$

Since

$$
P(C_i=k)=\phi_k
$$

and

\mathcal{N}(x_i\mid\mu_k,\Sigma_k),
$$

the marginal density becomes

\sum_{k=1}^{K}
\phi_k
\mathcal{N}(x_i\mid\mu_k,\Sigma_k)
}
$$

where the multivariate Gaussian density is

\frac{1}
{(2\pi)^{d/2}|\Sigma_k|^{1/2}}
\exp
\left[
-\frac12
(x_i-\mu_k)^T
\Sigma_k^{-1}
(x_i-\mu_k)
\right].
$$

Why is this called a Gaussian mixture density?

The marginal density is a weighted sum of $K$ Gaussian densities.

Each Gaussian represents one component or cluster, while $\phi_k$ specifies the relative contribution of component $k$.

Therefore,

\phi_1\mathcal{N}_1(x)
+
\phi_2\mathcal{N}_2(x)
+\cdots+
\phi_K\mathcal{N}_K(x).
$$

It is therefore called a Gaussian Mixture Model (GMM).

Unlike a single Gaussian distribution, a Gaussian mixture can represent:

multiple modes or peaks,
clusters with different centers,
clusters with different variances,
clusters with different orientations and correlations.
2. Deriving the Posterior Cluster Probability

For a fixed observed point $x_i$, we want to determine the probability that it belongs to cluster $k$:

$$
P(C_i=k\mid X_i=x_i).
$$

Using Bayes' rule,

\frac{
p(x_i\mid C_i=k)P(C_i=k)
}{
p(x_i)
}.
$$

Since

\sum_{j=1}^{K}
p(x_i\mid C_i=j)P(C_i=j),
$$

we obtain

\frac{
p(x_i\mid C_i=k)P(C_i=k)
}{
\displaystyle
\sum_{j=1}^{K}
p(x_i\mid C_i=j)P(C_i=j)
}.
$$

For continuous $X_i$, the terms involving $X_i=x_i$ are interpreted as probability densities, rather than probabilities at a single point.

Substituting

$$
P(C_i=k)=\phi_k
$$

and

\mathcal{N}(x_i\mid\mu_k,\Sigma_k),
$$

gives

\frac{
\phi_k
\mathcal{N}(x_i\mid\mu_k,\Sigma_k)
}{
\displaystyle
\sum_{j=1}^{K}
\phi_j
\mathcal{N}(x_i\mid\mu_j,\Sigma_j)
}
}
$$

This probability is called the responsibility of cluster $k$ for observation $x_i$:

P(C_i=k\mid X_i=x_i)
}
$$

The responsibilities satisfy

$$
0\leq\gamma_{ik}\leq1
$$

and

$$
\sum_{k=1}^{K}\gamma_{ik}=1.
$$

Interpretation

The responsibility $\gamma_{ik}$ is a posterior probability because it combines:

Prior cluster probability

$$
\phi_k=P(C_i=k)
$$

with the likelihood

$$
\mathcal{N}(x_i\mid\mu_k,\Sigma_k).
$$

Thus,

$$
\text{Posterior}
\propto
\text{Prior}
\times
\text{Likelihood}.
$$

A large $\gamma_{ik}$ means that, after observing $x_i$, there is strong evidence that the point belongs to cluster $k$.

3. One-Hot Encoding of the Latent Cluster Variable

Define the one-hot latent vector

\begin{bmatrix}
Z_{i1}\
Z_{i2}\
\vdots\
Z_{iK}
\end{bmatrix},
$$

where

\begin{cases}
1, & C_i=k,\
0, & \text{otherwise}.
\end{cases}
$$

Because $Z_{ik}$ is an indicator random variable,

$$
E[Z_{ik}\mid X_i=x_i]
$$

can be written as

1\cdot P(Z_{ik}=1\mid X_i=x_i)
+
0\cdot P(Z_{ik}=0\mid X_i=x_i).
$$

Therefore,

P(Z_{ik}=1\mid X_i=x_i).
$$

But

$$
Z_{ik}=1
\iff
C_i=k.
$$

Hence,

P(C_i=k\mid X_i=x_i)

\gamma_{ik}
}
$$

Therefore,

\begin{bmatrix}
\gamma_{i1}\
\gamma_{i2}\
\vdots\
\gamma_{iK}
\end{bmatrix}
}
$$

Thus, the soft cluster assignment vector is precisely

$$
\boxed{
E[Z_i\mid X_i=x_i]
}
$$

This is an important interpretation of GMM clustering: instead of immediately assigning a point to one cluster, the model computes the expected values of its latent cluster membership indicators.

4. From Soft Assignment to Hard Clustering

The soft assignment for observation $x_i$ is

\begin{bmatrix}
\gamma_{i1}\
\gamma_{i2}\
\vdots\
\gamma_{iK}
\end{bmatrix}.
$$

For example,

\begin{bmatrix}
0.10\
0.65\
0.25
\end{bmatrix}
$$

means that the observation has:

10% posterior probability of belonging to cluster 1,
65% posterior probability of belonging to cluster 2,
25% posterior probability of belonging to cluster 3.

This is called soft clustering because uncertainty about cluster membership is retained.

A hard assignment is obtained using

\arg\max_{1\leq k\leq K}
\gamma_{ik}
}
$$

For the previous example,

$$
\hat C_i=2.
$$

Difference between soft and hard clustering
Soft clustering

Soft clustering preserves all posterior probabilities:

$$
(\gamma_{i1},\ldots,\gamma_{iK}).
$$

It therefore represents uncertainty and overlapping clusters.

Hard clustering

Hard clustering assigns each point to only the cluster with the largest responsibility.

It retains only

$$
\arg\max_k\gamma_{ik}
$$

and discards the probabilities of the other clusters.

Therefore, GMM is fundamentally a soft probabilistic clustering method, although hard labels can be generated from its posterior probabilities.

5. Conditional Expectation of the Observation Given the Cluster

By assumption,

$$
X_i\mid C_i=k
\sim
\mathcal{N}(\mu_k,\Sigma_k).
$$

The expected value of a Gaussian random vector is its mean.

Therefore,

\mu_k
}
$$

Hence, $\mu_k$ represents the expected location of observations generated from cluster $k$.

For this reason, $\mu_k$ can be interpreted as the center of cluster $k$.

Comparison of the Two Conditional Expectations

The first conditional expectation is

\begin{bmatrix}
\gamma_{i1}\
\vdots\
\gamma_{iK}
\end{bmatrix}.
$$

This answers:

Given an observed point $x_i$, how likely is it to belong to each cluster?

Therefore, it gives the soft cluster membership of an observed point.

The second conditional expectation is

\mu_k.
$$

This answers:

Given that an observation belongs to cluster $k$, where do we expect that observation to be located?

Therefore, it gives the mean spatial location or center of cluster $k$.

Thus,

$$
\boxed{
E[Z_i\mid X_i=x_i]
\rightarrow
\text{membership information}
}
$$

while

$$
\boxed{
E[X_i\mid C_i=k]
\rightarrow
\text{cluster location information}.
}
$$

6. Complete-Data Likelihood

Suppose that both the observations and their latent labels were known.

The complete-data likelihood is

\prod_{i=1}^{n}
\prod_{k=1}^{K}
\left[
\phi_k
\mathcal{N}(x_i\mid\mu_k,\Sigma_k)
\right]^{z_{ik}}.
$$

Taking the logarithm,

\log
\left[
\prod_{i=1}^{n}
\prod_{k=1}^{K}
\left(
\phi_k
\mathcal{N}(x_i\mid\mu_k,\Sigma_k)
\right)^{z_{ik}}
\right].
$$

Using

\sum_j\log a_j,
$$

we obtain

\sum_{i=1}^{n}
\sum_{k=1}^{K}
z_{ik}
\log
\left[
\phi_k
\mathcal{N}(x_i\mid\mu_k,\Sigma_k)
\right].
$$

Using

$$
\log(ab)=\log a+\log b,
$$

the complete-data log-likelihood becomes

\sum_{i=1}^{n}
\sum_{k=1}^{K}
z_{ik}
\left[
\log\phi_k
+
\log
\mathcal{N}(x_i\mid\mu_k,\Sigma_k)
\right]
}
$$

Why would this be easy to maximize if $z_{ik}$ were known?

If the indicators were known, we would know exactly which observations belong to each cluster.

For cluster $k$, only observations having

$$
z_{ik}=1
$$

would contribute to its parameter estimates.

We could then directly calculate:

the proportion of observations in each cluster,
the sample mean of each cluster,
the sample covariance of each cluster.

Thus, parameter estimation would reduce to ordinary Gaussian maximum-likelihood estimation for separately labeled groups.

The difficulty is that the $z_{ik}$ values are hidden.

7. EM Interpretation

Since the latent variables $Z_i$ are not observed, the Expectation-Maximization algorithm works with their conditional expectations.

During the E-step,

$$
z_{ik}
\quad\longrightarrow\quad
E[Z_{ik}\mid X_i=x_i].
$$

Since

\gamma_{ik},
$$

we replace

$$
\boxed{
z_{ik}\longrightarrow\gamma_{ik}.
}
$$

The expected complete-data log-likelihood becomes

\sum_{i=1}^{n}
\sum_{k=1}^{K}
\gamma_{ik}
\left[
\log\phi_k
+
\log
\mathcal{N}(x_i\mid\mu_k,\Sigma_k)
\right]
}
$$

More precisely, at each EM iteration, the responsibilities are calculated using the current parameter estimates.

The E-step therefore calculates

\frac{
\phi_k
\mathcal{N}(x_i\mid\mu_k,\Sigma_k)
}{
\displaystyle
\sum_{j=1}^{K}
\phi_j
\mathcal{N}(x_i\mid\mu_j,\Sigma_j)
}.
$$

Why is the E-step a conditional update?

Before observing $x_i$, the cluster probability is

$$
P(C_i=k)=\phi_k.
$$

After observing $x_i$, this prior belief is updated using Bayes' rule:

$$
\phi_k
\quad\longrightarrow\quad
\gamma_{ik}.
$$

Thus,

$$
\text{Prior membership probability}
+
\text{Observed data}
\rightarrow
\text{Posterior membership probability}.
$$

Therefore, the E-step is a conditional probabilistic update of cluster membership.

8. Derivation of the GMM Parameter Updates

Define the effective number of observations belonging to cluster $k$ as

\sum_{i=1}^{n}
\gamma_{ik}.
}
$$

Because $\gamma_{ik}$ can lie between 0 and 1, $N_k$ is an effective or fractional cluster size.

8.1 Updating the Mixture Weights

The part of $Q$ involving $\phi_k$ is

\sum_{k=1}^{K}
N_k\log\phi_k.
$$

We must maximize this subject to

$$
\sum_{k=1}^{K}\phi_k=1.
$$

Introduce a Lagrange multiplier $\lambda$:

\sum_{k=1}^{K}
N_k\log\phi_k
+
\lambda
\left(
\sum_{k=1}^{K}\phi_k-1
\right).
$$

Differentiate with respect to $\phi_k$:




$$

Therefore,

-\frac{N_k}{\lambda}.
$$

Using

$$
\sum_{k=1}^{K}\phi_k=1
$$

and

$$
\sum_{k=1}^{K}N_k=n,
$$

we obtain

\frac{N_k}{n}.
}
$$

Thus, the updated mixture weight is the effective fraction of observations belonging to cluster $k$.

8.2 Updating the Cluster Mean

The Gaussian contribution involving $\mu_k$ is maximized by solving




$$

Therefore,

\mu_k
\sum_{i=1}^{n}
\gamma_{ik}.
$$

Since

\sum_{i=1}^{n}\gamma_{ik},
$$

we obtain

\frac{1}{N_k}
\sum_{i=1}^{n}
\gamma_{ik}x_i.
}
$$

Therefore, the updated cluster center is a responsibility-weighted mean of all observations.

8.3 Updating the Covariance Matrix

Similarly, maximizing $Q$ with respect to $\Sigma_k$ gives

\frac{1}{N_k}
\sum_{i=1}^{n}
\gamma_{ik}
(x_i-\mu_k^{\text{new}})
(x_i-\mu_k^{\text{new}})^T
}
$$

Therefore, the covariance matrix is a responsibility-weighted covariance around the newly updated mean.

Final M-Step Updates

The complete standard GMM updates are therefore

\sum_{i=1}^{n}\gamma_{ik}
}
$$

\frac{N_k}{n}
}
$$

\frac{1}{N_k}
\sum_{i=1}^{n}
\gamma_{ik}x_i
}
$$

and

\frac{1}{N_k}
\sum_{i=1}^{n}
\gamma_{ik}
(x_i-\mu_k^{\text{new}})
(x_i-\mu_k^{\text{new}})^T.
}
$$

Interpretation of $\gamma_{ik}$ as a Fractional Membership Weight

Unlike hard clustering, GMM does not initially treat an observation as belonging completely to only one cluster.

For example, suppose

(0.20,0.70,0.10).
$$

Then observation $x_i$ contributes:

weight $0.20$ to cluster 1,
weight $0.70$ to cluster 2,
weight $0.10$ to cluster 3.

Therefore, $\gamma_{ik}$ acts as a fractional membership weight when computing the new means, covariances, and mixture proportions.

9. Interpretation: GMM as Repeated Conditional Updating

Gaussian mixture clustering can be viewed as a repeated process of conditional probabilistic updating.

The mixture weight

$$
\phi_k
$$

represents the prior probability that an observation belongs to cluster $k$ before considering its observed feature values.

The Gaussian density

$$
\mathcal{N}(x_i\mid\mu_k,\Sigma_k)
$$

measures how compatible the observed point $x_i$ is with cluster $k$.

Bayes' rule combines these quantities to calculate

P(C_i=k\mid X_i=x_i),
$$

which is the posterior probability of cluster membership after observing $x_i$.

The complete soft assignment is

\begin{bmatrix}
\gamma_{i1}\
\vdots\
\gamma_{iK}
\end{bmatrix}.
$$

The E-step calculates these conditional expectations using the current cluster parameters.

The M-step then uses the responsibilities as fractional membership weights to update

$$
\phi_k,\qquad
\mu_k,\qquad
\Sigma_k.
$$

The new parameters change the Gaussian component distributions, which in turn change the responsibilities during the next E-step.

Therefore, EM repeatedly performs

$$
\boxed{
\text{Estimate memberships}
\rightarrow
\text{Update cluster parameters}
\rightarrow
\text{Re-estimate memberships}
\rightarrow\cdots
}
$$

until the model converges.

Hence, Gaussian mixture clustering is a form of probabilistic clustering based on conditional expectations of latent cluster membership variables.

10. Computational Simulation and Out-of-Sample Validation

We now implement a two-dimensional GMM using the credit-card customer dataset.

The two selected continuous variables are:

PURCHASES: customer purchase amount.
CREDIT_LIMIT: customer's credit limit.

The purpose is to fit a

$$
K=3
$$

component Gaussian mixture model and connect the computational results with the theoretical responsibilities

P(C_i=k\mid X_i=x_i).
$$

In [11]:
!pip -q install kagglehub

In [12]:
import kagglehub
import pandas as pd
from pathlib import Path

# Download Kaggle dataset
dataset_path = kagglehub.dataset_download( "arjunbhasin2013/ccdata" )
print("Dataset downloaded to:")
print(dataset_path)

# Find the CSV file
csv_files = list(Path(dataset_path).glob("*.csv"))
if not csv_files:
  raise FileNotFoundError(
      "No CSV file was found in the downloaded dataset." )

# Prefer CC GENERAL.csv if available
csv_path = next( (
    file for file in csv_files
    if file.name.lower() == "cc general.csv"
    ),
    csv_files[0] )

print("\nUsing file:")
print(csv_path)

# Load data
df = pd.read_csv(csv_path)
print("\nDataset shape:", df.shape)
display(df.head())
print("\nSelected feature summary:")
display( df[["PURCHASES", "CREDIT_LIMIT"]].describe() )
print("\nMissing values:")
print( df[["PURCHASES", "CREDIT_LIMIT"]] .isnull() .sum() )

Using Colab cache for faster access to the 'ccdata' dataset.
Dataset downloaded to:
/kaggle/input/ccdata

Using file:
/kaggle/input/ccdata/CC GENERAL.csv

Dataset shape: (8950, 18)


,CUST_ID,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,C10001,40.900749,0.818182,95.40,0.00,95.4,0.000000,0.166667,0.000000,0.083333,0.000000,0,2,1000.0,201.802084,139.509787,0.000000,12
1,C10002,3202.467416,0.909091,0.00,0.00,0.0,6442.945483,0.000000,0.000000,0.000000,0.250000,4,0,7000.0,4103.032597,1072.340217,0.222222,12
2,C10003,2495.148862,1.000000,773.17,773.17,0.0,0.000000,1.000000,1.000000,0.000000,0.000000,0,12,7500.0,622.066742,627.284787,0.000000,12
3,C10004,1666.670542,0.636364,1499.00,1499.00,0.0,205.788017,0.083333,0.083333,0.000000,0.083333,1,1,7500.0,0.000000,NaN,0.000000,12
4,C10005,817.714335,1.000000,16.00,16.00,0.0,0.000000,0.083333,0.083333,0.000000,0.000000,0,1,1200.0,678.334763,244.791237,0.000000,12



Selected feature summary:


,PURCHASES,CREDIT_LIMIT
count,8950.000000,8949.000000
mean,1003.204834,4494.449450
std,2136.634782,3638.815725
min,0.000000,50.000000
25%,39.635000,1600.000000
50%,361.280000,3000.000000
75%,1110.130000,6500.000000
max,49039.570000,30000.000000



Missing values:
PURCHASES       0
CREDIT_LIMIT    1
dtype: int64


10.2 GMMFinancialSegmenter Class

The class below performs:

Missing-value removal.

80% training and 20% test splitting.

Standardization using training-set statistics only.

GMM fitting with $K=3$.

EM convergence reporting.

Out-of-sample average log-likelihood evaluation.

Responsibility calculation.

Three interactive Plotly visualizations.

In [25]:
  import numpy as np
  import pandas as pd
  import plotly.express as px
  import plotly.graph_objects as go
  from sklearn.model_selection import train_test_split
  from sklearn.preprocessing import StandardScaler
  from sklearn.mixture import GaussianMixture

  class GMMFinancialSegmenter:
    def __init__( self, n_components=3, test_size=0.20, random_state=42 ):
      """
      Two-dimensional Gaussian Mixture Model segmenter.
      """
      self.n_components = n_components
      self.test_size = test_size
      self.random_state = random_state
      self.features = None
      self.scaler = StandardScaler()
      self.gmm = GaussianMixture(
          n_components=n_components,
          covariance_type="full",
          n_init=10,
          max_iter=500,
          reg_covar=1e-6,
          random_state=random_state )

      self.X_train_raw = None
      self.X_test_raw = None
      self.X_train_scaled = None
      self.X_test_scaled = None
      self.train_labels = None
      self.test_labels = None
      self.train_responsibilities = None
      self.test_responsibilities = None
      self.test_log_likelihood = None

    # --------------------------------------------------
    # 1. Prepare data
    # --------------------------------------------------

    def prepare_data( self, dataframe, features=(
        "PURCHASES", "CREDIT_LIMIT") ):
      self.features = list(features)

      # Check columns
      missing_columns = [
          col for col in self.features if col not in dataframe.columns
          ]

      if missing_columns:
        raise ValueError(
            f"Missing required columns: " f"{missing_columns}" )

      # Select only the two required features
      data = (
          dataframe[self.features]
          .replace([np.inf, -np.inf], np.nan)
          .dropna() .astype(float) )

      print( "Number of usable observations:", len(data) )

          # IMPORTANT:
          # Split before fitting the scaler to avoid
          # information leakage from the test data.
      (
            self.X_train_raw,
            self.X_test_raw

        ) = train_test_split(
            data,
            test_size=self.test_size,
            random_state=self.random_state,
            shuffle=True )

      # Fit scaler ONLY on training data
      self.X_train_scaled = (
          self.scaler.fit_transform(
              self.X_train_raw ) )

      # Apply the same transformation to test data
      self.X_test_scaled = (
          self.scaler.transform( self.X_test_raw ) )

      print( "Training observations:", len(self.X_train_raw) )
      print( "Test observations:", len(self.X_test_raw) )
      return self

    # --------------------------------------------------
    # 2. Fit GMM using EM
    # --------------------------------------------------
    def fit(self):
      if self.X_train_scaled is None:
        raise RuntimeError( "Call prepare_data() before fit()." )

      self.gmm.fit( self.X_train_scaled )

      # Hard assignments
      self.train_labels = self.gmm.predict( self.X_train_scaled )
      self.test_labels = self.gmm.predict( self.X_test_scaled )

      # Soft assignments / responsibilities
      self.train_responsibilities = ( self.gmm.predict_proba(
          self.X_train_scaled ) )
      self.test_responsibilities = ( self.gmm.predict_proba(
          self.X_test_scaled ) )

      # Average log-likelihood on unseen data
      self.test_log_likelihood = ( self.gmm.score( self.X_test_scaled ) )

      print( "EM converged:", self.gmm.converged_ )
      print( "Number of EM iterations:", self.gmm.n_iter_ )
      print( "Average test log-likelihood:", round(
          self.test_log_likelihood, 4 ) )
      print("\nLearned mixture weights:")

      for k, weight in enumerate( self.gmm.weights_, start=1 ):
        print( f"Cluster {k}: " f"{weight:.4f}" )
      return self

    # --------------------------------------------------
    # 3. Empirical density heatmap
    # --------------------------------------------------
    def plot_empirical_density(self):
      if self.X_train_raw is None:
        raise RuntimeError( "Data has not been prepared." )

      fig = px.density_heatmap(
          self.X_train_raw,
          x=self.features[0],
          y=self.features[1],
          nbinsx=60,
          nbinsy=60,
          marginal_x="histogram",
          marginal_y="histogram",
          title=( "Empirical 2D Density of " "Training Data" ) )

      fig.update_layout( template="plotly_white" )
      return fig

    # --------------------------------------------------
    # 4. Create responsibility grid
    # --------------------------------------------------
    def _responsibility_grid(
        self,
        grid_size=220,
        quantile_limits=(0.01, 0.99) ):

      if self.train_responsibilities is None:
        raise RuntimeError( "Fit the model before " "creating responsibility maps." )
      # Corrected indentation starts here
      combined = pd.concat( [ self.X_train_raw, self.X_test_raw ], axis=0 )
      x_feature = self.features[0]
      y_feature = self.features[1]
      q_low, q_high = quantile_limits
      x_min = combined[ x_feature ].quantile(q_low)
      x_max = combined[ x_feature ].quantile(q_high)
      y_min = combined[ y_feature ].quantile(q_low)
      y_max = combined[ y_feature ].quantile(q_high)

      # Small padding
      x_pad = 0.05 * (x_max - x_min)
      y_pad = 0.05 * (y_max - y_min)
      x_values = np.linspace(
          max(0, x_min - x_pad),
          x_max + x_pad, grid_size )
      y_values = np.linspace(
          max(0, y_min - y_pad),
          y_max + y_pad, grid_size )
      xx, yy = np.meshgrid( x_values, y_values )

      # Grid in original feature units
      grid_raw = pd.DataFrame( {
          x_feature: xx.ravel(),
          y_feature: yy.ravel() } )

      # Transform using SAME training scaler
      grid_scaled = self.scaler.transform( grid_raw )

      # Compute:
      #
      # [γ_i1, γ_i2, ..., γ_iK]
      #
      responsibilities = ( self.gmm.predict_proba( grid_scaled ) )
      # Largest posterior probability
      max_responsibility = ( responsibilities.max(axis=1) )
      # Hard cluster from argmax
      hard_cluster = ( responsibilities.argmax(axis=1) )
      return ( xx, yy, max_responsibility.reshape( xx.shape ), hard_cluster.reshape( xx.shape ) )

    # --------------------------------------------------
    # 5. Common assignment plot
    # --------------------------------------------------

    def _assignment_plot( self, raw_data, labels, title ):
      ( xx, yy, max_gamma, hard_grid ) = self._responsibility_grid()
      fig = go.Figure()

      # Continuous maximum responsibility map
      fig.add_trace(
          go.Contour(
              x=xx[0],
              y=yy[:, 0],
              z=max_gamma,
              contours=dict(
                  start=1 / self.n_components,
                  end=1.0, size=0.05 ),
              colorbar=dict( title="max γ" ),
              opacity=0.75, name=( "Maximum Posterior " "Responsibility" )
          ) )

      # Overlay observed points
      fig.add_trace( go.Scatter(
          x=raw_data[ self.features[0] ],
          y=raw_data[ self.features[1] ],
          mode="markers",
          marker=dict(
              size=5,
              color=labels,
              colorscale="Turbo",
              line=dict( width=0.3 ) ),
          text=[ f"Cluster {c + 1}" for c in labels ],
          hovertemplate=(
              self.features[0]
              + ": %{x:.2f}<br>"
              + self.features[1]
              + ": %{y:.2f}<br>"
              + "%^{text}" + "<extra></extra>" ),
          name="Observed Points" ) )

      fig.update_layout(
          title=title,
          xaxis_title=self.features[0],
          yaxis_title=self.features[1],
          template="plotly_white" )
      return fig


    # --------------------------------------------------
    # 6. Training assignment plot
    # --------------------------------------------------

    def plot_training_assignments(self):
      return self._assignment_plot(
          self.X_train_raw,
          self.train_labels, (
              "Training Data: GMM Assignments "
              "and Maximum Posterior " "Responsibility" ) )

    # --------------------------------------------------
    # 7. Test assignment plot
    # --------------------------------------------------
    def plot_test_assignments(self):
      return self._assignment_plot( self.X_test_raw, self.test_labels, (
          "Out-of-Sample Test Data: "
          "GMM Assignments and "
          "Maximum Posterior Responsibility" ) )

    # --------------------------------------------------
    # 8. Display component means in original units
    # --------------------------------------------------
    def display_cluster_centers(self):
      centers_original = (
          self.scaler.inverse_transform( self.gmm.means_ ) )
      centers = pd.DataFrame( centers_original, columns=self.features )
      centers.index = [
          f"Cluster {k + 1}" for k in range( self.n_components ) ]

      return centers

In [27]:
  segmenter = GMMFinancialSegmenter(
      n_components=3,
      test_size=0.20,
      random_state=42 )

  segmenter.prepare_data(
      df,
      features=( "PURCHASES", "CREDIT_LIMIT" ) )
  segmenter.fit()

Number of usable observations: 8949
Training observations: 7159
Test observations: 1790
EM converged: True
Number of EM iterations: 20
Average test log-likelihood: -1.6888

Learned mixture weights:
Cluster 1: 0.1041
Cluster 2: 0.4428
Cluster 3: 0.4531


In [29]:
  cluster_centers = ( segmenter.display_cluster_centers() )
  display(cluster_centers)

,PURCHASES,CREDIT_LIMIT
Cluster 1,4631.641306,9578.341851
Cluster 2,183.625369,2070.207836
Cluster 3,918.775706,5744.381470


In [34]:
  fig1 = segmenter.plot_empirical_density()
  fig1.show()

  fig2 = (
      segmenter
      .plot_training_assignments()
      )
  fig2.show()

  fig3 = ( segmenter .plot_test_assignments() )
  fig3.show()

TypeError: cannot unpack non-iterable NoneType object

The model calculates

test_score = (
    segmenter.test_log_likelihood
)

print(
    "Average log-likelihood "
    "on unseen test data:",
    test_score
)

Mathematically, the average test log-likelihood is

$$
\frac{1}{n_{\text{test}}}
\sum_{i=1}^{n_{\text{test}}}
\log
\left[
\sum_{k=1}^{K}
\phi_k
\mathcal{N}
(
x_i^{test}
\mid
\mu_k,\Sigma_k
)
\right].
$$

This measures how much probability density the fitted GMM assigns to unseen observations.

A larger, or less negative, average log-likelihood generally means that the fitted density explains the unseen observations better when comparing models on the same data representation.

It should not be interpreted as classification accuracy because clustering has no true class labels in this application.

10.9 Direct Examination of Responsibilities

The first few responsibility vectors can be displayed using:

In [36]:
  responsibility_df = pd.DataFrame(
      segmenter.train_responsibilities,
      columns=[ "γ_cluster_1", "γ_cluster_2", "γ_cluster_3" ] )
  display( responsibility_df.head(10) )

,γ_cluster_1,γ_cluster_2,γ_cluster_3
0,0.013622,2.767924e-08,0.986378
1,0.048552,1.076604e-11,0.951448
2,0.020617,2.079435e-03,0.977304
3,0.009742,3.113775e-01,0.678881
4,0.028985,4.570849e-10,0.971015
5,0.471778,7.016247e-37,0.528222
6,0.000794,9.722350e-01,0.026971
7,0.011440,9.452975e-02,0.894030
8,0.013368,3.715767e-02,0.949474
9,0.012271,6.283800e-02,0.924891


For every observation,




$$

This can be verified computationally:

print(
    responsibility_df
    .sum(axis=1)
    .head(10)
)

For example, a hypothetical result such as

$$
[0.02,;0.91,;0.07]
$$

indicates strong membership in cluster 2.

In contrast,

$$
[0.42,;0.46,;0.12]
$$

shows uncertainty between clusters 1 and 2.

The hard assignment would choose cluster 2 in both cases, but only the soft responsibility vector preserves the uncertainty in the second case.

10.10 Relationship Between the Contour Map and the Conditional Expectation

From the theoretical derivation,

\begin{bmatrix}
\gamma_{i1}\
\gamma_{i2}\
\gamma_{i3}
\end{bmatrix}.
$$

The computational visualization extends this idea from only the observed points to every coordinate on a fine grid.

For each grid location

$$
x_{\text{grid}},
$$

the fitted GMM calculates

\begin{bmatrix}
\gamma_1(x_{\text{grid}})\
\gamma_2(x_{\text{grid}})\
\gamma_3(x_{\text{grid}})
\end{bmatrix}.
$$

The contour background displays

$$
\max_k
\gamma_k(x_{\text{grid}}).
$$

Therefore, the background is a continuous visualization of the posterior cluster-membership probabilities derived analytically.

High maximum responsibility means that one component strongly dominates:

$$
E[Z\mid X=x]
\approx
[1,0,0]^T,
$$

or a permutation of this vector.

Lower maximum responsibility indicates that probability is distributed across multiple clusters, for example

$$
E[Z\mid X=x]
\approx
[0.45,0.48,0.07]^T.
$$

These lower-confidence areas correspond to soft cluster boundaries or ambiguity regions.

The scatter-point colors show the hard decision

$$
\arg\max_k\gamma_k(x),
$$

whereas the continuous background preserves information about how certain that decision is.

10.11 Overall Evaluation

The three visualizations illustrate three different aspects of Gaussian mixture clustering.

Empirical Density Heatmap

The density heatmap displays the raw distribution of PURCHASES and CREDIT_LIMIT.

It allows us to visually inspect whether the customer population contains multiple regions of concentration and whether a single Gaussian distribution would be too restrictive.

Training Assignment Plot

The training plot shows how the fitted Gaussian components divide the feature space.

Individual observations receive hard labels for visualization, but the background demonstrates that the underlying GMM calculation is probabilistic.

The transition regions between strongly assigned areas demonstrate that GMM does not fundamentally create rigid boundaries. Instead, posterior membership probabilities vary continuously across the feature space.

Test Assignment Plot

The test plot applies the learned Gaussian mixture to observations that were not used during model training.

This demonstrates out-of-sample conditional updating:

$$
x_{\text{new}}
\rightarrow
P(C=k\mid X=x_{\text{new}}).
$$

A new customer's financial measurements can therefore be inserted into the trained model, and the model calculates a responsibility vector without refitting the GMM.

The average test log-likelihood provides an additional quantitative measure of how well the learned probability density generalizes to the unseen data.

Final Conclusion

Gaussian Mixture Model clustering can be understood naturally as repeated conditional updating of latent cluster memberships.

Before observing a data point, the mixture weight

$$
\phi_k
$$

acts as the prior probability of membership in cluster $k$.

The Gaussian density

$$
\mathcal{N}(x_i\mid\mu_k,\Sigma_k)
$$

measures the compatibility of the observed point with that cluster.

Bayes' rule combines these quantities to obtain

P(C_i=k\mid X_i=x_i),
$$

the posterior cluster probability.

Because

\gamma_{ik},
$$

the complete soft assignment is

[
\gamma_{i1},
\ldots,
\gamma_{iK}
]^T.
}
$$

The E-step calculates these posterior membership probabilities.

The M-step then updates

$$
\phi_k,\qquad
\mu_k,\qquad
\Sigma_k
$$

using the responsibilities as fractional weights.

These two steps are repeated until convergence.

The computational contour map makes this theoretical result visible. Every point in the two-dimensional feature space has a complete posterior membership vector, while the hard cluster label simply selects its largest component.

Therefore,

$$
\boxed{
\text{GMM clustering is probabilistic clustering based on conditional expectations of latent cluster membership variables.}
}
$$